In [ ]:
from pathlib import Path
import os
import warnings

import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from shapely.prepared import prep as shapely_prep

warnings.filterwarnings("ignore")

ROOT = Path(".")
OUT = ROOT / "data_out"
RAW = ROOT / "data_raw"
OUT.mkdir(exist_ok=True)
RAW.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
GRID_08 = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")
POI_08 = Path("../08 POI Demand/data_out/sz_poi.gpkg")

GRID_SIZE = 0.005
PROVIDER_CSV = RAW / "provider_poi_shenzhen.csv"

### 安全提示（必读）

- **永远不要把 Key 发到聊天、邮件、截图或 Git 仓库**。若已外泄，请到[高德控制台](https://console.amap.com/)**删除该 Key 或重置密钥**，再新建一个。
- 本笔记本**只**从环境变量 `AMAP_KEY` 读取；本地运行前在终端执行：`export AMAP_KEY='（你的新 key）'`（不要把引号里的内容提交到版本库）。


In [ ]:
# ── 与 08 相同的深圳格网（h3_id 对齐用 08 的 demand 文件，H3 res=8） ──
shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
sz_prep = shapely_prep(shenzhen_geom)

if GRID_08.exists():
    grid_ref = gpd.read_file(GRID_08)
    print(f"08 基准格网: {len(grid_ref):,} cells (h3_id 与 08 一致)")
else:
    grid_ref = None
    print("未找到 08 sz_demand_grid.gpkg，将仅统计点集自身。")

In [ ]:
# ── 基准：00 / 2026.01 → 08 输出 ──
if POI_08.exists():
    base = gpd.read_file(POI_08)
    print("=== 08 基准（00/02-poi-aoi 本地 SHP 八类 → sz_poi.gpkg）===")
    print(f"  点数: {len(base):,}")
    print(f"  poi_type 分布:\n{base['poi_type'].value_counts().to_string()}")
else:
    base = None
    print("无 sz_poi.gpkg")

In [ ]:
# ── 图商全类目：从 CSV 读入（自行准备） ──
# CSV 列名可映射：经度列 lon/lng/x，纬度 lat/y
def load_provider_csv(path: Path) -> gpd.GeoDataFrame:
    df = pd.read_csv(path)
    lon_c = next((c for c in df.columns if c.lower() in ("lon", "lng", "longitude", "x")), None)
    lat_c = next((c for c in df.columns if c.lower() in ("lat", "latitude", "y")), None)
    if not lon_c or not lat_c:
        raise ValueError("CSV 需包含经度/纬度列（如 lon,lat 或 lng,lat）")
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_c].astype(float), df[lat_c].astype(float)),
        crs=4326,
    )
    gdf = gpd.clip(gdf, shenzhen_geom)
    # 去重：名称 + 坐标（与 08 同量级）
    gdf["_lon"] = gdf.geometry.x.round(5)
    gdf["_lat"] = gdf.geometry.y.round(5)
    name_col = next((c for c in gdf.columns if "name" in c.lower() or "名称" in c), None)
    if name_col:
        gdf = gdf.drop_duplicates(subset=[name_col, "_lon", "_lat"])
    else:
        gdf = gdf.drop_duplicates(subset=["_lon", "_lat"])
    return gdf.drop(columns=["_lon", "_lat"], errors="ignore")


if PROVIDER_CSV.exists():
    prov = load_provider_csv(PROVIDER_CSV)
    print("=== 图商/全类目 CSV ===")
    print(f"  深圳内去重点数: {len(prov):,}")
    if base is not None:
        print(f"  vs 08 点数比: {len(prov) / len(base):.2f}x")
else:
    prov = None
    print(f"未找到 {PROVIDER_CSV.name}，跳过图商统计。请将全类目导出保存为该路径。")

In [ ]:
# ── 映射到与 08 类似的 demand_group（可按 type 字段自定义） ──
# 高德 type 常为 "餐饮服务;中餐厅" 这类字符串，需自建关键词规则
def map_amap_type_to_group(type_str: str) -> str:
    if not isinstance(type_str, str):
        return "other"
    s = type_str
    if "餐饮" in s or "美食" in s:
        return "food"
    if "购物" in s or "商场" in s or "超市" in s:
        return "retail"
    if "医疗" in s or "医院" in s or "诊所" in s:
        return "medical"
    if "科教" in s or "学校" in s or "文化" in s:
        return "education"
    if "商务" in s or "住宅" in s or "写字楼" in s:
        return "office"
    if "体育" in s or "休闲" in s or "运动" in s:
        return "leisure"
    if "风景" in s or "旅游" in s or "公园" in s:
        return "scenic"
    if "生活" in s or "服务" in s:
        return "service"
    return "other"


if prov is not None and grid_ref is not None:
    type_col = next((c for c in prov.columns if c.lower() in ("type", "types", "poi_type", "大类")), None)
    if type_col:
        prov["demand_group"] = prov[type_col].astype(str).map(map_amap_type_to_group)
    else:
        prov["demand_group"] = "other"
    joined = gpd.sjoin(prov, grid_ref[["h3_id", "geometry"]], how="inner", predicate="within")
    pivot = joined.groupby(["h3_id", "demand_group"]).size().unstack(fill_value=0)
    for c in ["food", "retail", "medical", "education", "office", "leisure", "scenic", "service"]:
        if c not in pivot.columns:
            pivot[c] = 0
    pivot = pivot.rename(columns={c: f"{c}_count" for c in pivot.columns if not str(c).endswith("_count")})
    pivot["poi_total"] = pivot.sum(axis=1)
    WEIGHTS = {
        "food_count": 0.25,
        "retail_count": 0.20,
        "medical_count": 0.15,
        "office_count": 0.15,
        "education_count": 0.05,
        "service_count": 0.05,
        "leisure_count": 0.05,
        "scenic_count": 0.05,
    }
    pivot["demand_pressure"] = 0.0
    for k, w in WEIGHTS.items():
        if k in pivot.columns:
            pivot["demand_pressure"] += w * pivot[k]
    mx = pivot["demand_pressure"].max()
    if mx > 0:
        pivot["demand_pressure_norm"] = pivot["demand_pressure"] / mx
    cmp_grid = grid_ref[["h3_id", "geometry"]].merge(pivot.reset_index(), on="h3_id", how="left").fillna(0)
    cmp_grid.to_file(OUT / "sz_demand_grid_provider_mapped.gpkg", driver="GPKG")
    print("已写:", OUT / "sz_demand_grid_provider_mapped.gpkg")
    print("other 类未计入 demand_pressure 权重时，可在 WEIGHTS 中扩展。")
else:
    print("跳过格网聚合（缺少 prov 或 08 格网）")

In [ ]:
# ── 与 08 demand_pressure 逐格相关（需两边 h3_id 一致） ──
if grid_ref is not None and "demand_pressure" in grid_ref.columns and (OUT / "sz_demand_grid_provider_mapped.gpkg").exists():
    g08 = grid_ref.set_index("h3_id")["demand_pressure"]
    gpv = gpd.read_file(OUT / "sz_demand_grid_provider_mapped.gpkg").set_index("h3_id")["demand_pressure"]
    common = g08.align(gpv, join="inner")
    c = np.corrcoef(common[0], common[1])[0, 1]
    print(f"demand_pressure 皮尔逊相关(同 h3_id): {c:.3f}")
else:
    print("无法算相关：缺少 08 demand_pressure 或 provider 格网输出")

In [ ]:
key = os.environ.get("AMAP_KEY", "")
if not key:
    print("未设置环境变量 AMAP_KEY，跳过 API 示例。")
else:
    # 深圳市民中心附近 1km 内「餐厅」示例（单页最多 20+ 条，视接口而定）
    url = "https://restapi.amap.com/v3/place/around"
    params = {
        "key": key,
        "location": "114.057868,22.543099",
        "keywords": "餐厅",
        "types": "050000",
        "radius": "1000",
        "offset": "25",
        "page": "1",
        "extensions": "base",
    }
    r = requests.get(url, params=params, timeout=15)
    data = r.json()
    print("status:", data.get("status"), "count:", data.get("count"))
    pois = data.get("pois") or []
    print("本页 POI 条数:", len(pois))
    if pois:
        print("示例:", pois[0].get("name"), pois[0].get("type"))

In [ ]:
ak = os.environ.get("BAIDU_AK", "")
if not ak:
    print("未设置环境变量 BAIDU_AK，跳过百度 API 示例。")
else:
    url = "https://api.map.baidu.com/place/v2/search"
    params = {
        "query": "餐厅",
        "region": "深圳",
        "output": "json",
        "ak": ak,
        "page_size": 20,
        "page_num": 0,
        "scope": 1,
    }
    r = requests.get(url, params=params, timeout=15)
    data = r.json()
    st = data.get("status")
    print("status:", st, "message:", data.get("message", ""))
    results = data.get("results") or []
    print("本页 POI 条数:", len(results))
    if results:
        p0 = results[0]
        loc = p0.get("location", {})
        print("示例:", p0.get("name"), "|", loc.get("lng"), loc.get("lat"))

## 高德 POI → `provider_poi_shenzhen.csv`（合规与用法）

1. **密钥**：**不要**把 `key` 写进笔记本或提交到 Git。使用环境变量，例如终端先执行：`export AMAP_KEY='你的key'`（Windows 可用「系统环境变量」或 `set AMAP_KEY=...`）。若 Key 曾在聊天/截图中泄露，请在高德控制台**立即重置**。
2. **接口**：以官网为准，常用为 **地点搜索 `v3/place/text`**（`city` + `types` 类目码 + `page`/`offset` 分页）。类目码见高德文档「POI 分类码」。
3. **配额**：全深圳、全类目会消耗大量配额；可先**少量 types + 少量 max_pages** 试跑，再按需扩大。
4. **输出**：`place/text` 与 `place/polygon` 两格脚本均可写入 `data_raw/provider_poi_shenzhen.csv`（后者会**覆盖**同名文件；按需只跑一格或改文件名）。
5. **分块（多边形）**：当 `city + types` 仍担心漏点或想按空间切块时，可用官网 **`v3/place/polygon`**：对深圳边界包络盒划分矩形，逐块分页请求；块与块重叠会导致重复，需用 `id` 或 `name+lon+lat` **去重**（见再下一格）。


In [ ]:
# ── 高德 place/text：按「城市 + types」分页拉取并保存 CSV（key 仅来自环境变量）──
import time

OUT_CSV = RAW / "provider_poi_shenzhen.csv"
key = os.environ.get("AMAP_KEY", "").strip()

# 高德 POI 大类码节选；全量对比可逐步补全（见官方 POI 分类码表）
AMAP_TYPES = [
    "050000",  # 餐饮服务
    "060000",  # 购物服务
    "070000",  # 生活服务
    "080000",  # 体育休闲服务
    "090000",  # 医疗保健服务
    "100000",  # 住宿服务
    "110000",  # 风景名胜
    "120000",  # 商务住宅
    "130000",  # 政府机构及社会团体
    "140000",  # 科教文化服务
    "150000",  # 交通设施服务
    "160000",  # 金融保险服务
    "170000",  # 公司企业
    "180000",  # 道路附属设施
    "190000",  # 地名地址信息
    "200000",  # 公共设施
]


def _fetch_place_text(types_code: str, city: str = "深圳", max_pages: int = 100, sleep_s: float = 0.2):
    url = "https://restapi.amap.com/v3/place/text"
    rows = []
    offset = 25
    for page in range(1, max_pages + 1):
        params = {
            "key": key,
            "city": city,
            "citylimit": "true",
            "types": types_code,
            "offset": str(offset),
            "page": str(page),
            "extensions": "all",
        }
        r = requests.get(url, params=params, timeout=30)
        data = r.json()
        if str(data.get("status")) != "1":
            break
        pois = data.get("pois") or []
        for p in pois:
            loc = (p.get("location") or "").strip()
            parts = loc.split(",")
            try:
                lon = float(parts[0])
                lat = float(parts[1])
            except (ValueError, IndexError):
                lon, lat = None, None
            rows.append(
                {
                    "id": p.get("id"),
                    "name": p.get("name"),
                    "type": p.get("type"),
                    "address": p.get("address"),
                    "lon": lon,
                    "lat": lat,
                    "types_code": types_code,
                }
            )
        if len(pois) < offset:
            break
        time.sleep(sleep_s)
    return rows


if not key:
    print("未设置 AMAP_KEY。请在运行前 export AMAP_KEY='...'，不要把 key 写进笔记本。")
else:
    # 首次试跑：只拉前 3 个大类、每类最多 10 页，避免一次打满配额；需要全量时改大 DEMO_TYPES / DEMO_MAX_PAGES
    DEMO_TYPES = AMAP_TYPES[:3]
    DEMO_MAX_PAGES = 10
    all_rows = []
    for tc in DEMO_TYPES:
        print(f"types={tc} …")
        all_rows.extend(_fetch_place_text(tc, max_pages=DEMO_MAX_PAGES))
    df_api = pd.DataFrame(all_rows)
    if len(df_api) == 0:
        print("未拉到数据：检查 key、配额或缩小 citylimit。")
    else:
        if df_api["id"].notna().any():
            df_api = df_api.drop_duplicates(subset=["id"])
        else:
            df_api = df_api.drop_duplicates(subset=["name", "lon", "lat"])
        df_api.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
        print(f"已写入 {OUT_CSV}  行数={len(df_api):,}（可按需增大 DEMO_TYPES / DEMO_MAX_PAGES / AMAP_TYPES）")


In [ ]:
# ── 可选：高德 v3/place/polygon — 按矩形瓦片 + 分页（与 place/text 二选一或合并去重）──
# 须先运行上文「深圳边界」格，得到 shenzhen_geom。Key 仅来自环境变量。
import time

OUT_CSV_POLY = RAW / "provider_poi_shenzhen.csv"
key_poly = os.environ.get("AMAP_KEY", "").strip()


def _box_to_amap_polygon(b) -> str:
    x0, y0, x1, y1 = b.bounds
    return f"{x0},{y0}|{x1},{y0}|{x1},{y1}|{x0},{y1}|{x0},{y0}"


def _fetch_place_polygon(poly_str: str, types_code: str = "", max_pages: int = 20, sleep_s: float = 0.25):
    url = "https://restapi.amap.com/v3/place/polygon"
    rows = []
    offset = 25
    for page in range(1, max_pages + 1):
        params = {
            "key": key_poly,
            "polygon": poly_str,
            "types": types_code,
            "offset": str(offset),
            "page": str(page),
            "extensions": "all",
        }
        r = requests.get(url, params=params, timeout=30)
        data = r.json()
        if str(data.get("status")) != "1":
            break
        pois = data.get("pois") or []
        for p in pois:
            loc = (p.get("location") or "").strip()
            parts = loc.split(",")
            try:
                lon = float(parts[0])
                lat = float(parts[1])
            except (ValueError, IndexError):
                lon, lat = None, None
            rows.append(
                {
                    "id": p.get("id"),
                    "name": p.get("name"),
                    "type": p.get("type"),
                    "address": p.get("address"),
                    "lon": lon,
                    "lat": lat,
                    "polygon_tile": poly_str[:40] + "…",
                }
            )
        if len(pois) < offset:
            break
        time.sleep(sleep_s)
    return rows


if not key_poly:
    print("未设置 AMAP_KEY，跳过 place/polygon。")
else:
    try:
        sg = shenzhen_geom
    except NameError:
        print("请先运行加载深圳边界的单元格（shenzhen_geom）。")
        sg = None
    if sg is not None:
        # 试跑：粗瓦片 + 少量块 + 少量页；扩大前请估算配额（块数 × 每块页数）
        STEP = 0.12
        DEMO_MAX_TILES = 8
        PAGES_PER_TILE = 8
        TYPES_FILTER = "050000"
        minx, miny, maxx, maxy = sg.bounds
        tiles = []
        x = minx
        while x < maxx:
            y = miny
            while y < maxy:
                b = box(x, y, min(x + STEP, maxx), min(y + STEP, maxy))
                if b.intersects(sg):
                    tiles.append(b)
                y += STEP
            x += STEP
        tiles = tiles[:DEMO_MAX_TILES]
        all_poly = []
        for i, b in enumerate(tiles):
            ps = _box_to_amap_polygon(b)
            print(f"tile {i+1}/{len(tiles)} …")
            all_poly.extend(_fetch_place_polygon(ps, types_code=TYPES_FILTER, max_pages=PAGES_PER_TILE))
        df_poly = pd.DataFrame(all_poly)
        if len(df_poly) == 0:
            print("polygon 未返回数据：检查 key、polygon 参数或 types。")
        else:
            df_poly = df_poly.dropna(subset=["lon", "lat"])
            gtmp = gpd.GeoDataFrame(
                df_poly,
                geometry=gpd.points_from_xy(df_poly["lon"], df_poly["lat"]),
                crs=4326,
            )
            gtmp = gpd.clip(gtmp, sg)
            df_poly = pd.DataFrame(gtmp.drop(columns="geometry"))
            if df_poly["id"].notna().any():
                df_poly = df_poly.drop_duplicates(subset=["id"])
            else:
                df_poly = df_poly.drop_duplicates(subset=["name", "lon", "lat"])
            df_poly.to_csv(OUT_CSV_POLY, index=False, encoding="utf-8-sig")
            print(f"已写入 {OUT_CSV_POLY}  行数={len(df_poly):,}（增大 STEP 降瓦片数；减小 STEP 更密覆盖；改 DEMO_MAX_TILES / PAGES_PER_TILE / TYPES_FILTER）")
